# 02 · Cosmology

Along a photon path the scale factor obeys da/dx = (H0/c) √(Ω_r + Ω_m a + Ω_k a² + Ω_Λ a⁴) / √(1 − k x²)
(notes `sec_cosmology.tex`, Eq. dadx_general). `Cosmology` holds a batch of backgrounds and integrates it with
`diffrax`. The skewers (03) use it to solve a(x) along every path. Runtime: ~10 s.

**Requires:** nothing.  
**Independent of:** lines, skewers, fields.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import (use_style, ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, LABELS, FIGSIZES,
                                      add_panel_letters, plot_dag, plot_spectra, zoom_inset)
use_style()                                        # the dark, colour-blind-safe style of every figure (skewerjax.utils.plotting)
%matplotlib inline

In [ ]:
plot_dag(('cosmology',)); plt.show()

In [ ]:
from skewerjax.core import Cosmology, phys
h = phys.H0_over_c                                                  # H0 / c [1/Mpc]
cosmo = Cosmology(H0=jnp.full(4, phys.H0), Omega_r_0=jnp.array([1.0, 0.0, 0.0, 9.1436e-5]), Omega_m_0=jnp.array([0.0, 1.0, 0.0, 0.3089]),
                  Omega_k_0=jnp.zeros(4), Omega_lambda_0=jnp.array([0.0, 0.0, 1.0, 0.6911]))
x_f, x_b = np.linspace(0, 1 / h - 1e-6, 600), np.linspace(0, -2 / h + 1e-6, 600)
a = np.concatenate([np.asarray(cosmo.solve_a_of_x(jnp.ones(4), jnp.asarray(x_b)))[:, ::-1], np.asarray(cosmo.solve_a_of_x(jnp.ones(4), jnp.asarray(x_f)))], 1)
x = np.concatenate([x_b[::-1], x_f])
exact = {"radiation": 1 + h * x, "matter": (1 + 0.5 * h * x) ** 2, "Lambda": 1 / (1 - h * x)}
fig, (ax, axr) = plt.subplots(2, 1, figsize=(13, 9), sharex=True, gridspec_kw={"height_ratios": [3, 1]}, layout="constrained")
for i, (name, c) in enumerate(zip(["radiation", "matter", "Lambda", "Planck 2018"], [TOL["red"], TOL["blue"], TOL["green"], FG])):
    ax.plot(x, a[i], color=c, label=name)
    if name in exact:
        ok = np.isfinite(exact[name]) & (exact[name] > 0)
        ax.plot(x[ok], exact[name][ok], color=c, **MS["reference"])
        axr.plot(x[ok], a[i][ok] / exact[name][ok] - 1, color=c)
ax.plot(0, 1, "x", color=FG, ms=12, mew=3); ax.annotate("observer today", (0, 1), (300, 0.3), arrowprops=dict(arrowstyle="->"), fontsize=15)
ax.set(yscale="log", ylim=(1e-3, 1e2), ylabel="scale factor $a$"); ax.legend(loc="lower right")
axr.set(xlabel="comoving distance $x$ [Mpc] (towards the future)", ylabel="residual", yscale="symlog", ylim=(-1e-3, 1e-3))
axr.axhline(0, color=FG, lw=1); add_panel_letters([ax, axr])
fig.suptitle("a(x) along a light ray: the ODE solution follows the closed forms (thick, light)", fontsize=22); plt.show()

The default tolerances (`rtol = atol = 1e-4`) give a(x) to ~10⁻⁴; `solve_a_of_x(..., rtol=1e-10, atol=1e-12)` to ~10⁻⁹ (`scripts/validation/check_cosmology.py`). The parameters are pytree leaves: `jax.grad` of any result with respect to Ω_m works.

## Summary

- a(x) along a light ray is solved with diffrax and matches the closed forms to the ODE tolerance.
- The cosmological parameters are pytree leaves (batches of backgrounds, gradients).
- Checks: `scripts/validation/check_cosmology.py`. Next: the skewers that use it (03).